# Without Pointer-Generator (reference notebook)

Original exploratory notebook (outputs stripped), kept for reference. The production code in `src/rare_words_summ/` is the canonical version; this notebook's `USE_POINTER` flag is unified there as `model.use_pointer`.

In [ ]:
!pip install datasets rouge-score nltk tqdm --quiet

In [ ]:
import os, math, re, copy, json, warnings
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


import nltk
from nltk import ngrams as nltk_ngrams
from nltk.tokenize import sent_tokenize, word_tokenize

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from rouge_score import rouge_scorer
torch.backends.cudnn.benchmark = True   # put near the top, after imports
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
dataset = load_dataset('cnn_dailymail', '3.0.0')
df      = pd.DataFrame(dataset['train'])
df      = df.rename(columns={'article':'Text', 'highlights':'Summary'})
df      = df.dropna(subset=['Text','Summary'])
df      = df[(df['Text'].str.strip()!='') & (df['Summary'].str.strip()!='')]

# subsample so it finishes on a T4 (remove .sample(...) to use all 287k)
df      = df.sample(n=60000, random_state=42).reset_index(drop=True)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

In [ ]:
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))


def remove_stopwords_tokens(tokens):
    return [t for t in tokens if t.lower() not in STOPWORDS and t.isalpha()]

# Corpus-level 3-gram frequency (paper: across whole training set)
corpus_ngram_counts = Counter()
for text in tqdm(train_df['Text'], desc='Building 3-gram dictionary'):
    tokens = word_tokenize(text.lower())
    tokens = remove_stopwords_tokens(tokens)
    corpus_ngram_counts.update(list(nltk_ngrams(tokens, 3)))

corpus_ngram_counts = {k:v for k,v in corpus_ngram_counts.items() if v > 1}
print(f'Unique 3-grams (freq>1): {len(corpus_ngram_counts):,}')

# Paper Eq 3: score = 1/log(occ) + shift
raw_scores       = {g: 1.0/math.log(occ) for g,occ in corpus_ngram_counts.items()}
shift            = 1.0 - np.mean(list(raw_scores.values()))
ngram_score_dict = {g: s+shift for g,s in raw_scores.items()}
max_score        = max(ngram_score_dict.values())
print(f'Shift: {shift:.4f} | Max score: {max_score:.4f}')

In [ ]:
def score_sentence(sentence, ngram_dict, max_sc):
    tokens = remove_stopwords_tokens(word_tokenize(sentence.lower()))
    if len(tokens) < 3:
        return 0.0
    scores = [ngram_dict.get(g, max_sc) for g in nltk_ngrams(tokens, 3)]
    return float(np.mean(scores))

def extractive_summary(text, ngram_dict, max_sc, max_tokens):
    sentences = sent_tokenize(text)
    if not sentences:
        return text
    scored        = [(i,s,score_sentence(s,ngram_dict,max_sc)) for i,s in enumerate(sentences)]
    scored_sorted = sorted(scored, key=lambda x: x[2], reverse=True)
    selected_idx, token_count = set(), 0
    for idx, sent, _ in scored_sorted:
        n = len(word_tokenize(sent.lower()))  # match downstream tokenizer
        if token_count + n <= max_tokens:
            selected_idx.add(idx); token_count += n
        if token_count >= max_tokens:
            break
    # Keep original document order
    selected = [s for i,s,_ in scored if i in selected_idx]
    return ' '.join(selected) if selected else sentences[0]

for split_name, split_df in [('train',train_df),('val',val_df),('test',test_df)]:
    for mt in [400]:
        split_df[f'ExtractedText_{mt}'] = [
            extractive_summary(t, ngram_score_dict, max_score, mt)
            for t in tqdm(split_df['Text'], desc=f'{split_name} {mt} tokens', leave=True)
        ]

# Save CSVs
os.makedirs('/kaggle/working', exist_ok=True)
for split_name, split_df in [('train',train_df),('val',val_df),('test',test_df)]:
    for mt in [400]:
        out = split_df[[f'ExtractedText_{mt}','Summary']].copy()
        out.columns = ['Text','Summary']
        out.to_csv(f'/kaggle/working/{split_name}_{mt}.csv', index=False)
print('Extractive stage complete. CSVs saved.')

In [ ]:
PAD_TOKEN='<pad>'; UNK_TOKEN='<unk>'; SOS_TOKEN='<sos>'; EOS_TOKEN='<eos>'
SPECIAL = [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]

class Vocabulary:
    def __init__(self, max_vocab=15000):
        self.max_vocab=max_vocab; self.word2idx={}; self.idx2word={}
        self.word_freq=Counter(); self.freq_scores={}; self.max_freq_sc=3.0

    def build(self, texts):
        for t in texts:
            self.word_freq.update(word_tokenize(t.lower()))
        common = self.word_freq.most_common(self.max_vocab - len(SPECIAL))
        for i,tok in enumerate(SPECIAL):
            self.word2idx[tok]=i; self.idx2word[i]=tok
        for i,(w,_) in enumerate(common, len(SPECIAL)):
            self.word2idx[w]=i; self.idx2word[i]=w
        raw = {w:1.0/math.log(occ) for w,occ in self.word_freq.items() if occ>1}
        if raw:
            sh = 1.0 - np.mean(list(raw.values()))
            self.freq_scores  = {w:s+sh for w,s in raw.items()}
            self.max_freq_sc  = max(self.freq_scores.values())
        print(f'Vocab size: {len(self.word2idx):,}')

    def get_freq_score(self, word):
        return self.freq_scores.get(word, self.max_freq_sc)

    @property
    def pad_idx(self): return self.word2idx[PAD_TOKEN]
    @property
    def unk_idx(self): return self.word2idx[UNK_TOKEN]
    @property
    def sos_idx(self): return self.word2idx[SOS_TOKEN]
    @property
    def eos_idx(self): return self.word2idx[EOS_TOKEN]
    def __len__(self): return len(self.word2idx)

vocab = Vocabulary(max_vocab=15000)
vocab.build(list(train_df['ExtractedText_400']) + list(train_df['Summary']))
VOCAB_SIZE = len(vocab)

In [ ]:
class LegalSumDataset(Dataset):
    def __init__(self, df, vocab, max_src=512, max_tgt=150, text_col='Text'):
        self.vocab = vocab
        self.max_src = max_src; self.max_tgt = max_tgt
        df = df.reset_index(drop=True)
        # Pre-tokenize ONCE (word_tokenize is the slow part) -> cache token lists.
        # __getitem__ now just looks these up, so the GPU stops waiting on the CPU.
        self.src_toks = [
            word_tokenize(str(t).lower())[:max_src]
            for t in tqdm(df[text_col], desc=f'Tokenizing src ({text_col})', leave=False)
        ]
        self.tgt_toks = [
            word_tokenize(str(t).lower())[:max_tgt-2]
            for t in tqdm(df['Summary'], desc='Tokenizing tgt', leave=False)
        ]

    def __len__(self): return len(self.src_toks)

    def __getitem__(self, idx):
        src_toks = self.src_toks[idx]      # cached — no word_tokenize here
        tgt_toks = self.tgt_toks[idx]

        # OOV tracking for pointer-generator
        oov_list, oov_map = [], {}
        for t in src_toks:
            if t not in self.vocab.word2idx and t not in oov_map:
                oov_map[t] = VOCAB_SIZE + len(oov_list)
                oov_list.append(t)

        src_ids  = [self.vocab.word2idx.get(t, self.vocab.unk_idx) for t in src_toks]
        src_ext  = [oov_map.get(t, self.vocab.word2idx.get(t, self.vocab.unk_idx)) for t in src_toks]
        src_freq = [self.vocab.get_freq_score(t) for t in src_toks]
        tgt_in   = [self.vocab.sos_idx] + [self.vocab.word2idx.get(t, self.vocab.unk_idx) for t in tgt_toks]
        tgt_out  = [oov_map.get(t, self.vocab.word2idx.get(t, self.vocab.unk_idx)) for t in tgt_toks] + [self.vocab.eos_idx]

        pad_src = self.max_src - len(src_ids)
        src_ids  += [self.vocab.pad_idx]*pad_src
        src_ext  += [self.vocab.pad_idx]*pad_src
        src_freq += [0.0]*pad_src
        pad_tgt = self.max_tgt - len(tgt_in)
        tgt_in  += [self.vocab.pad_idx]*pad_tgt
        tgt_out += [self.vocab.pad_idx]*(self.max_tgt - len(tgt_out))

        return (
            torch.tensor(src_ids,  dtype=torch.long),
            torch.tensor(src_ext,  dtype=torch.long),
            torch.tensor(tgt_in,   dtype=torch.long),
            torch.tensor(tgt_out,  dtype=torch.long),
            torch.tensor(src_freq, dtype=torch.float),
            len(oov_list)
        )

def collate_fn(batch):
    src_ids,src_ext,tgt_ids,tgt_ext,src_freq,oov_lens = zip(*batch)
    return (torch.stack(src_ids), torch.stack(src_ext),
            torch.stack(tgt_ids), torch.stack(tgt_ext),
            torch.stack(src_freq), max(oov_lens))

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1024, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class FrequencyAwareAttention(nn.Module):
    """Encoder self-attention with M2 frequency boosting (paper Eq 4)."""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads=n_heads; self.d_k=d_model//n_heads; self.d_model=d_model
        self.W_q=nn.Linear(d_model,d_model); self.W_k=nn.Linear(d_model,d_model)
        self.W_v=nn.Linear(d_model,d_model); self.W_o=nn.Linear(d_model,d_model)
        self.W_freq=nn.Linear(1,1,bias=False)
        nn.init.ones_(self.W_freq.weight)
        self.dropout=nn.Dropout(dropout)
    def split(self,x):
        B,L,_=x.shape
        return x.view(B,L,self.n_heads,self.d_k).transpose(1,2)
    def forward(self, q, k, v, mask=None, freq=None):
        B=q.size(0)
        Q,K,V=self.split(self.W_q(q)),self.split(self.W_k(k)),self.split(self.W_v(v))
        sc = torch.matmul(Q,K.transpose(-2,-1)) / math.sqrt(self.d_k)
        if freq is not None:
            # FIX: softplus keeps the freq multiplier > 0 so W_freq can't flip
            # the sign of every attention logit during training.
            fs = F.softplus(self.W_freq(freq.unsqueeze(1).unsqueeze(2).unsqueeze(-1))).squeeze(-1)
            sc = sc * fs
        if mask is not None: sc=sc.masked_fill(mask==0,-1e4)
        at = self.dropout(torch.softmax(sc,dim=-1))
        out= torch.matmul(at,V).transpose(1,2).contiguous().view(B,-1,self.d_model)
        return self.W_o(out)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads=n_heads; self.d_k=d_model//n_heads; self.d_model=d_model
        self.W_q=nn.Linear(d_model,d_model); self.W_k=nn.Linear(d_model,d_model)
        self.W_v=nn.Linear(d_model,d_model); self.W_o=nn.Linear(d_model,d_model)
        self.dropout=nn.Dropout(dropout)
    def split(self,x):
        B,L,_=x.shape
        return x.view(B,L,self.n_heads,self.d_k).transpose(1,2)
    def forward(self, q, k, v, mask=None):
        B=q.size(0)
        Q,K,V=self.split(self.W_q(q)),self.split(self.W_k(k)),self.split(self.W_v(v))
        sc=torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(self.d_k)
        if mask is not None: sc=sc.masked_fill(mask==0,-1e4)
        at=self.dropout(torch.softmax(sc,dim=-1))
        out=torch.matmul(at,V).transpose(1,2).contiguous().view(B,-1,self.d_model)
        return self.W_o(out), at


class FeedForward(nn.Module):
    def __init__(self,d,dff,dr=0.1):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(d,dff),nn.ReLU(),nn.Dropout(dr),nn.Linear(dff,d))
    def forward(self,x): return self.net(x)


class EncoderLayer(nn.Module):
    def __init__(self,d,h,dff,dr=0.1):
        super().__init__()
        self.attn=FrequencyAwareAttention(d,h,dr); self.ff=FeedForward(d,dff,dr)
        self.n1=nn.LayerNorm(d); self.n2=nn.LayerNorm(d); self.dr=nn.Dropout(dr)
    def forward(self,x,mask,freq=None):
        x=self.n1(x+self.dr(self.attn(x,x,x,mask,freq)))
        return self.n2(x+self.dr(self.ff(x)))


class DecoderLayer(nn.Module):
    def __init__(self,d,h,dff,dr=0.1):
        super().__init__()
        self.s_attn=MultiHeadAttention(d,h,dr); self.c_attn=MultiHeadAttention(d,h,dr)
        self.ff=FeedForward(d,dff,dr)
        self.n1=nn.LayerNorm(d); self.n2=nn.LayerNorm(d); self.n3=nn.LayerNorm(d)
        self.dr=nn.Dropout(dr)
    def forward(self,x,enc,sm,tm):
        a1,_=self.s_attn(x,x,x,tm)
        x=self.n1(x+self.dr(a1))
        a2,cross=self.c_attn(x,enc,enc,sm)
        x=self.n2(x+self.dr(a2))
        return self.n3(x+self.dr(self.ff(x))), cross


class Encoder(nn.Module):
    def __init__(self,vs,d,nl,h,dff,ml,pi,dr=0.1):
        super().__init__()
        self.emb=nn.Embedding(vs,d,padding_idx=pi)
        self.pe=PositionalEncoding(d,ml,dr)
        self.layers=nn.ModuleList([EncoderLayer(d,h,dff,dr) for _ in range(nl)])
        self.norm=nn.LayerNorm(d)
    def forward(self,src,mask,freq=None):
        x=self.pe(self.emb(src))
        for l in self.layers: x=l(x,mask,freq)
        return self.norm(x)


class Decoder(nn.Module):
    def __init__(self,vs,d,nl,h,dff,ml,pi,dr=0.1):
        super().__init__()
        self.emb=nn.Embedding(vs,d,padding_idx=pi)
        self.pe=PositionalEncoding(d,ml,dr)
        self.layers=nn.ModuleList([DecoderLayer(d,h,dff,dr) for _ in range(nl)])
        self.norm=nn.LayerNorm(d)
    def forward(self,tgt,enc,sm,tm):
        x=self.pe(self.emb(tgt))
        cross=None
        for l in self.layers: x,cross=l(x,enc,sm,tm)
        return self.norm(x), cross


class PointerGeneratorLayer(nn.Module):
    """Real Pointer-Generator (See et al. 2017). Handles full sequence [B,T,d]."""
    def __init__(self, d_model):
        super().__init__()
        self.W_h=nn.Linear(d_model,1,bias=False)
        self.W_s=nn.Linear(d_model,1,bias=False)
        self.W_x=nn.Linear(d_model,1,bias=False)
        self.b_ptr=nn.Parameter(torch.zeros(1))

    def forward(self, context_vec, dec_state, dec_input,
                vocab_dist, attn_dist, src_ext_ids, max_oov):
        # All inputs are [B, T, d] — handles full sequence at once
        p_gen = torch.sigmoid(
            self.W_h(context_vec) + self.W_s(dec_state) +
            self.W_x(dec_input)   + self.b_ptr
        )  # [B, T, 1]

        B, T, V        = vocab_dist.shape
        ext_vocab_size = V + max_oov

        # Copy distribution over extended vocab -- PURE copy (starts at zeros).
        # FIX(Bug1): do NOT seed with vocab_dist; that double-counted mass and
        # made final_dist sum to (2 - p_gen) instead of 1, breaking the NLL loss.
        ext_dist = torch.zeros(B, T, ext_vocab_size, device=vocab_dist.device)
        src_ext  = src_ext_ids.unsqueeze(1).expand(B, T, -1)  # [B,T,src_len]
        ext_dist.scatter_add_(2, src_ext, attn_dist)

        # Vocab distribution over extended vocab
        vocab_ext = torch.zeros(B, T, ext_vocab_size, device=vocab_dist.device)
        vocab_ext[:,:,:V] = vocab_dist

        final_dist = p_gen * vocab_ext + (1 - p_gen) * ext_dist
        return final_dist, p_gen


class TransformerPG(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_layers=8, n_heads=8,
                 d_ff=1024, max_src_len=512, max_tgt_len=150, pad_idx=0, dropout=0.1,
                 use_pointer=True):
        super().__init__()
        self.encoder    = Encoder(vocab_size,d_model,n_layers,n_heads,d_ff,max_src_len,pad_idx,dropout)
        self.decoder    = Decoder(vocab_size,d_model,n_layers,n_heads,d_ff,max_tgt_len,pad_idx,dropout)
        self.vocab_proj = nn.Linear(d_model, vocab_size)
        self.pointer_gen= PointerGeneratorLayer(d_model)
        self.pad_idx    = pad_idx
        self.vocab_size = vocab_size
        self.use_pointer= use_pointer

    def make_src_mask(self,src):
        return (src!=self.pad_idx).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self,tgt):
        B,L=tgt.shape
        pm=(tgt!=self.pad_idx).unsqueeze(1).unsqueeze(2)
        sm=torch.tril(torch.ones(L,L,device=tgt.device)).bool()
        return pm & sm.unsqueeze(0).unsqueeze(0)

    def forward(self, src, src_ext_ids, tgt, freq_scores, max_oov):
        sm=self.make_src_mask(src); tm=self.make_tgt_mask(tgt)
        enc_out=self.encoder(src,sm,freq_scores)
        dec_out,cross=self.decoder(tgt,enc_out,sm,tm)

        vocab_logits = self.vocab_proj(dec_out)
        vocab_logits[:, :, self.pad_idx] = -1e9   # don't predict <pad>
        vocab_dist  = torch.softmax(vocab_logits, dim=-1)

        if not self.use_pointer:
            # No pointer-generator: plain abstractive transformer (paper M1/M2).
            return vocab_dist                        # [B, T, V]

        attn_dist   = cross.mean(dim=1)              # [B,T,src_len]
        # attention weights go through dropout in training, so row sums drift
        # from 1. Renormalize so the copy distribution stays valid.
        attn_dist   = attn_dist / attn_dist.sum(-1, keepdim=True).clamp(min=1e-9)
        context_vec = torch.bmm(attn_dist, enc_out)  # [B,T,d]
        dec_emb     = self.decoder.emb(tgt)          # [B,T,d]
        final_dist, p_gen = self.pointer_gen(
            context_vec, dec_out, dec_emb,
            vocab_dist, attn_dist, src_ext_ids, max_oov
        )
        return final_dist  # [B, T, ext_vocab]

print('Architecture defined.')

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
D_MODEL      = 256
N_LAYERS     = 8
N_HEADS      = 8
D_FF         = 1024
DROPOUT      = 0.1
MAX_SRC      = 512
MAX_TGT      = 150
BATCH_SIZE   = 8
EPOCHS       = 30
LR           = 1e-4
CLIP         = 1.0
WARMUP_STEPS = 500
PATIENCE     = 3      # early stopping patience
BEAM_SIZE    = 4
USE_POINTER  = False   # <-- set False to run WITHOUT the pointer-generator


# ── NLL Loss on extended vocab ────────────────────────────────────────────────
def compute_loss(final_dist, tgt_out, pad_idx):
    B, T, EV  = final_dist.shape
    log_dist  = torch.log(final_dist.clamp(min=1e-10))
    # Targets the model cannot emit (OOV ids when the pointer is OFF) -> <unk>.
    tgt = tgt_out.clone()
    tgt[tgt >= EV] = vocab.unk_idx
    nll       = -log_dist.gather(2, tgt.unsqueeze(2)).squeeze(2)  # [B,T]
    pad_mask  = (tgt_out == pad_idx)
    nll[pad_mask] = 0.0
    return nll.sum() / (~pad_mask).sum().float().clamp(min=1)


# ── LR warmup ─────────────────────────────────────────────────────────────────
def get_lr_lambda(step):
    step = step + 1
    return min(step / WARMUP_STEPS, 1.0)


# ── Train one epoch ───────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, device, epoch):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc=f'Epoch {epoch} [train]', leave=True)
    for src, src_ext, tgt_in, tgt_out, src_freq, max_oov in pbar:
        src,src_ext   = src.to(device), src_ext.to(device)
        tgt_in,tgt_out= tgt_in.to(device), tgt_out.to(device)
        src_freq      = src_freq.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            final_dist = model(src, src_ext, tgt_in, src_freq, max_oov)
            loss = compute_loss(final_dist.float(), tgt_out, vocab.pad_idx)  # .float() keeps loss in fp32
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    return total_loss / len(loader)


# ── Validation loss ───────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate_val_loss(model, loader, device):
    model.eval()
    total_loss = 0.0
    pbar = tqdm(loader, desc='Val loss', leave=False)
    for src, src_ext, tgt_in, tgt_out, src_freq, max_oov in pbar:
        src,src_ext    = src.to(device), src_ext.to(device)
        tgt_in,tgt_out = tgt_in.to(device), tgt_out.to(device)
        src_freq       = src_freq.to(device)
        final_dist     = model(src, src_ext, tgt_in, src_freq, max_oov)
        loss           = compute_loss(final_dist, tgt_out, vocab.pad_idx)
        total_loss    += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    return total_loss / len(loader)


# ── Beam search decode ────────────────────────────────────────────────────────
@torch.no_grad()
def beam_search_decode(model, src, src_ext, src_freq, vocab, max_len, device,
                        max_oov=0, beam_size=4, oov_words=None, length_alpha=0.7):
    """FIX(Bug2): copied OOV tokens are kept in the output (mapped back to the
    real source word) and fed to the decoder as <unk>. Adds length penalty so
    beams are not biased toward very short summaries."""
    model.eval()
    oov_words = oov_words or []
    src      = src.unsqueeze(0).to(device)
    src_ext  = src_ext.unsqueeze(0).to(device)
    src_freq = src_freq.unsqueeze(0).to(device)
    src_mask = (src != vocab.pad_idx).unsqueeze(1).unsqueeze(2)
    enc_out  = model.encoder(src, src_mask, src_freq)  # [1, src_len, d]

    # beam = (log_prob, out_ids, feed_ids)
    #   out_ids  : TRUE ids incl. extended OOV ids -> used for the final string
    #   feed_ids : embed-safe ids (OOV -> <unk>)   -> used as decoder input
    beams     = [(0.0, [vocab.sos_idx], [vocab.sos_idx])]
    completed = []

    def norm_score(item):
        return item[0] / (len(item[1]) ** length_alpha)

    for _ in range(max_len):
        candidates = []
        for log_prob, out_ids, feed_ids in beams:
            if out_ids[-1] == vocab.eos_idx:
                completed.append((log_prob, out_ids, feed_ids))
                continue
            tgt      = torch.tensor([feed_ids], device=device)   # feed embed-safe ids
            tgt_mask = model.make_tgt_mask(tgt)
            dec_out, ca = model.decoder(tgt, enc_out, src_mask, tgt_mask)
            attn_dist    = ca.mean(1)                  # [1,t,src_len]
            attn_dist    = attn_dist / attn_dist.sum(-1, keepdim=True).clamp(min=1e-9)
            vocab_logits = model.vocab_proj(dec_out[:,-1:])
            vocab_logits[:, :, vocab.pad_idx] = -1e9
            vocab_dist   = torch.softmax(vocab_logits, dim=-1)
            if getattr(model, 'use_pointer', True):
                context_vec  = torch.bmm(attn_dist[:,-1:], enc_out)
                dec_emb      = model.decoder.emb(tgt[:,-1:])
                final_dist, _ = model.pointer_gen(
                    context_vec, dec_out[:,-1:], dec_emb,
                    vocab_dist, attn_dist[:,-1:], src_ext, max_oov
                )  # [1,1,ext_vocab]
                log_probs   = torch.log(final_dist.squeeze(1).clamp(min=1e-10))  # [1,EV]
            else:
                log_probs   = torch.log(vocab_dist.squeeze(1).clamp(min=1e-10))  # [1,V]
            topk_lp, topk_ids = log_probs[0].topk(beam_size)
            for lp, tid in zip(topk_lp.tolist(), topk_ids.tolist()):
                feed_id = tid if tid < VOCAB_SIZE else vocab.unk_idx  # OOV -> <unk>
                candidates.append((log_prob + lp,
                                   out_ids  + [tid],
                                   feed_ids + [feed_id]))

        if not candidates:
            break
        beams = sorted(candidates, key=norm_score, reverse=True)[:beam_size]

        if len(completed) >= beam_size:
            break

    all_hyps = completed + beams
    best     = sorted(all_hyps, key=norm_score, reverse=True)[0]
    words    = []
    for tid in best[1][1:]:  # skip SOS, walk TRUE out_ids
        if tid == vocab.eos_idx:
            break
        if tid >= VOCAB_SIZE:                       # copied OOV -> real source word
            oov_i = tid - VOCAB_SIZE
            words.append(oov_words[oov_i] if oov_i < len(oov_words) else UNK_TOKEN)
        else:
            words.append(vocab.idx2word.get(tid, UNK_TOKEN))
    return ' '.join(words)


# ── ROUGE on test set (called only at end) ────────────────────────────────────
def evaluate_rouge(model, df, vocab, text_col, device):
    scorer_obj = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    r1,r2,rl   = 0.0,0.0,0.0
    n          = len(df)
    for i in tqdm(range(n), desc='Test ROUGE'):
        src_text = str(df.iloc[i][text_col])
        ref_text = str(df.iloc[i]['Summary'])
        tokens   = word_tokenize(src_text.lower())[:MAX_SRC]
        oov_list,oov_map=[],[]
        oov_map  = {}
        for t in tokens:
            if t not in vocab.word2idx and t not in oov_map:
                oov_map[t] = VOCAB_SIZE + len(oov_list)
                oov_list.append(t)
        src_ids  = [vocab.word2idx.get(t,vocab.unk_idx) for t in tokens]
        src_ext  = [oov_map.get(t,vocab.word2idx.get(t,vocab.unk_idx)) for t in tokens]
        src_freq = [vocab.get_freq_score(t) for t in tokens]
        pad      = MAX_SRC - len(src_ids)
        src_ids  += [vocab.pad_idx]*pad
        src_ext  += [vocab.pad_idx]*pad
        src_freq += [0.0]*pad
        pred = beam_search_decode(
            model,
            torch.tensor(src_ids,  dtype=torch.long),
            torch.tensor(src_ext,  dtype=torch.long),
            torch.tensor(src_freq, dtype=torch.float),
            vocab, MAX_TGT, device, len(oov_list), BEAM_SIZE,
            oov_words=oov_list
        )
        sc=scorer_obj.score(ref_text,pred)
        r1+=sc['rouge1'].fmeasure; r2+=sc['rouge2'].fmeasure; rl+=sc['rougeL'].fmeasure
    return {'rouge1':r1/n,'rouge2':r2/n,'rougeL':rl/n}

print('Training utilities defined.')

In [ ]:
train_ds_400 = LegalSumDataset(train_df, vocab, MAX_SRC, MAX_TGT, 'ExtractedText_400')
val_ds_400   = LegalSumDataset(val_df,   vocab, MAX_SRC, MAX_TGT, 'ExtractedText_400')

train_loader_400 = DataLoader(train_ds_400, batch_size=BATCH_SIZE, shuffle=True,collate_fn=collate_fn, num_workers=4, pin_memory=True)
val_loader_400   = DataLoader(val_ds_400, batch_size=BATCH_SIZE, shuffle=False,collate_fn=collate_fn, num_workers=4, pin_memory=True)

model_400     = TransformerPG(VOCAB_SIZE,D_MODEL,N_LAYERS,N_HEADS,D_FF,
                               MAX_SRC,MAX_TGT,vocab.pad_idx,DROPOUT,
                               use_pointer=USE_POINTER).to(DEVICE)
optimizer_400 = torch.optim.Adam(model_400.parameters(), lr=LR, betas=(0.9,0.98), eps=1e-9)
scheduler_400 = torch.optim.lr_scheduler.LambdaLR(optimizer_400, get_lr_lambda)

print(f'Parameters: {sum(p.numel() for p in model_400.parameters() if p.requires_grad):,}')
print(f'\n=== Training 400-token model (use_pointer={USE_POINTER}) ===')

best_val_loss_400 = float('inf')
patience_counter  = 0
history_400       = []



In [ ]:
import time
model_400.train()
it = iter(train_loader_400)
torch.cuda.synchronize()
t_data = t_fwd = t_bwd = 0.0
for step in range(20):
    t0 = time.time()
    batch = next(it)
    src, src_ext, tgt_in, tgt_out, src_freq, max_oov = [
        x.to(DEVICE) if torch.is_tensor(x) else x for x in batch
    ]
    torch.cuda.synchronize(); t1 = time.time()

    final = model_400(src, src_ext, tgt_in, src_freq, max_oov)
    loss  = compute_loss(final, tgt_out, vocab.pad_idx)
    torch.cuda.synchronize(); t2 = time.time()

    optimizer_400.zero_grad(); loss.backward()
    torch.cuda.synchronize(); t3 = time.time()

    t_data += t1 - t0; t_fwd += t2 - t1; t_bwd += t3 - t2

print(f'data {t_data:.2f}s | forward {t_fwd:.2f}s | backward {t_bwd:.2f}s')

In [ ]:
scaler = torch.cuda.amp.GradScaler()
for epoch in range(1, EPOCHS+1):
    train_loss = train_epoch(model_400, train_loader_400, optimizer_400,
                             scheduler_400, DEVICE, epoch)
    val_loss   = evaluate_val_loss(model_400, val_loader_400, DEVICE)
    current_lr = scheduler_400.get_last_lr()[0]
    history_400.append({'epoch':epoch,'train_loss':train_loss,'val_loss':val_loss})
    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | '
          f'LR: {current_lr:.2e}')

    if val_loss < best_val_loss_400:
        best_val_loss_400 = val_loss
        patience_counter  = 0
        torch.save(model_400.state_dict(), f'/kaggle/working/best_model_{"pg" if USE_POINTER else "nopg"}_400.pt')
        print(f'  ✓ Best val loss improved → checkpoint saved')
    else:
        patience_counter += 1
        print(f'  No improvement ({patience_counter}/{PATIENCE})')
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nBest Val Loss (400 tokens): {best_val_loss_400:.4f}')

In [ ]:
print('=== Final Test Set Evaluation ===')

for tok, model_obj in [(400, model_400)]:
    model_obj.load_state_dict(
        torch.load(f'/kaggle/working/best_model_{"pg" if USE_POINTER else "nopg"}_{tok}.pt', map_location=DEVICE)
    )
    rouge = evaluate_rouge(model_obj, test_df, vocab,
                           f'ExtractedText_{tok}', DEVICE)
    print(f'\n{tok} tokens (best checkpoint):')
    print(f'  ROUGE-1: {rouge["rouge1"]:.4f}')
    print(f'  ROUGE-2: {rouge["rouge2"]:.4f}')
    print(f'  ROUGE-L: {rouge["rougeL"]:.4f}')